# Stage 6, Project: Recommendation System

**Problem:** given a user's past movie ratings, recommend movies they haven't seen but are likely to rate highly.

**Dataset:** MovieLens ml-latest-small — 610 users, 9,742 movies, ~100k ratings (1-5 stars, half-star increments), plus genre tags.

New concepts this project introduces: the user-item matrix, matrix factorization (this is where PCA/SVD's eigenvector intuition from Stage 1/4 pays off directly), and ranking-based evaluation (precision@k) instead of the classification metrics from Stage 5.

## 1. Load and first look

Guiding questions:
- How many unique users and movies are there? How does that compare to the number of ratings — what fraction of all possible (user, movie) pairs actually have a rating?
- This fraction has a name: **sparsity**. A user-item matrix where almost every cell is empty is the defining challenge of recommendation systems — keep this number in mind, it explains why naive approaches struggle here.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TODO: load ml-latest-small/ratings.csv and ml-latest-small/movies.csv

url1 = "ml-latest-small/ratings.csv"
url2 = "ml-latest-small/movies.csv"

ratings = pd.read_csv(url1)
movies = pd.read_csv(url2)

ratings.head()
movies.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
# TODO: n_users, n_movies, n_ratings, sparsity = 1 - (n_ratings / (n_users * n_movies))
# print sparsity as a percentage


## 2. EDA

Guiding questions:
- Plot the distribution of ratings per movie. Are most movies rated by very few users, with a small number of blockbusters dominating? (This is the classic "long tail" pattern in recommendation data.)
- Plot the distribution of ratings per user. Do most users rate few movies, with a handful of power-raters?
- Plot the distribution of the rating values themselves (1-5 stars) — any skew?

In [ ]:
# TODO: ratings-per-movie distribution, ratings-per-user distribution, rating value distribution


## 3. Train/test split — different rules for recommenders

This is NOT the same as a normal random row split. If you randomly split individual ratings, some users or movies could end up with zero ratings in train, making them impossible to recommend for/evaluate at all.

The standard approach: for each user, hold out a small number of their ratings (e.g., their most recent, or a random 20%) for testing, keeping the rest in train. This guarantees every user has *some* training history.

Guiding question: why would a purely random split (ignoring per-user grouping) risk leaking information or producing an unfair evaluation here, in a way it didn't for Titanic or house prices?

In [ ]:
# TODO: implement a per-user split — for each userId, put ~80% of their ratings in train, ~20% in test
# hint: use groupby('userId') and sample/split within each group


## 4. Baseline: Popularity Recommender

The simplest possible recommender: recommend the same top-N most popular (highest average-rated, with a minimum rating-count cutoff to avoid one-rating flukes) movies to every user, regardless of their individual taste.

Guiding question: why is a minimum rating-count cutoff necessary here? (Think about a movie with a single 5-star rating vs. one with 500 ratings averaging 4.5 — which is the more trustworthy "good movie" signal?)

In [ ]:
# TODO: compute average rating per movie on TRAIN data only, filter to movies with e.g. >= 20 ratings,
# sort descending -> this ranked list is your baseline recommender for every user


## 5. Collaborative Filtering via Matrix Factorization

The core idea: build a user-item matrix (rows = users, columns = movies, values = ratings, mostly empty given your sparsity finding). Then factor that matrix into two smaller matrices — one representing users as vectors in a small "latent taste space," one representing movies the same way — such that multiplying them back together approximates the original ratings, filling in the empty cells with predictions.

This is the direct payoff of Stage 1/4's PCA review: you're finding a lower-dimensional representation that captures the main directions of variation in the data, same underlying idea as PCA, applied to a sparse ratings matrix instead of a dense feature table.

Guiding question: what might a "latent dimension" the model discovers actually represent, conceptually (you'll never see it labeled explicitly, but reason about what kind of pattern would make users and movies align along a shared axis)? E.g., could one latent dimension end up implicitly capturing something like "prefers action movies vs. prefers dramas"?

In [ ]:
# TODO: pivot TRAIN ratings into a user-item matrix (pd.pivot_table or pivot), fill missing with 0


In [ ]:
from sklearn.decomposition import TruncatedSVD

# TODO: fit TruncatedSVD (try n_components=20 to start) on the user-item matrix
# this gives you user vectors (transform output) and movie vectors (components_)
# reconstruct approximate ratings via user_vectors @ movie_vectors to get predicted scores for EVERY user-movie pair


## 6. Content-Based Filtering

A completely different approach: instead of using other users' ratings, recommend movies similar in *content* (genres) to what a given user has rated highly.

Steps: build a genre feature vector per movie (one-hot the pipe-separated genres column), compute cosine similarity between movies, then for a given user, recommend movies most similar to the ones they've rated highest.

Guiding question: what's a scenario where content-based filtering would work well but collaborative filtering would struggle? (Think about a brand new movie with zero ratings yet — this is called the "cold start" problem.)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# TODO: one-hot encode genres per movie (split on '|'), compute cosine similarity matrix between all movies


## 7. Evaluation: Precision@K and Recall@K

Standard classification metrics don't directly apply here — there's no single "correct" label per prediction. Instead, recommenders are evaluated by: generate the top-K recommendations for a user, then check how many of the movies they *actually rated highly in the test set* appear in that top-K list.

- **Precision@K** = (relevant items in your top-K) / K
- **Recall@K** = (relevant items in your top-K) / (total relevant items that exist for this user in test)

Guiding question: define "relevant" for this dataset — would you count any rating in the test set as relevant, or only ratings above some threshold (e.g., >= 4 stars)? Justify your choice.

In [ ]:
# TODO: implement precision@K and recall@K, evaluate the popularity baseline, collaborative filtering, and content-based approaches
# average the per-user precision@K and recall@K across all users for each approach


## 8. Error analysis

Pick a handful of individual users. Look at what they actually rated highly (train set) versus what each approach recommended. Where do the recommendations make intuitive sense, and where do they clearly miss?

In [ ]:
# TODO: for 2-3 sample users, print their top-rated train movies alongside top recommendations from each approach


## Wrap-up

Write a short summary:
- Which approach won on precision@K / recall@K — popularity baseline, collaborative filtering, or content-based?
- Did collaborative filtering meaningfully beat the popularity baseline? If the gap is smaller than you expected, what does that suggest about this dataset's sparsity?
- What would a production recommendation system likely do differently from any single approach here (hint: think about combining approaches, and how you'd handle a brand-new user with zero ratings)?

_Your summary here._